<div style="display: flex; align-items: center; justify-content: center; gap: 20px;">
    <img src="https://logo-marque.com/wp-content/uploads/2021/03/Kayak-Logo.png" alt="Image Gauche" style="width: 400px; height: auto;">
    <div style="font-size: 26px; text-align: center;">
        <h1>
            <span style="font-weight: bold; color: #FF6A06; text-decoration: underline;">
                Kayak Project
            </span>
        </h1>
    </div>
    <img src="https://logo-marque.com/wp-content/uploads/2021/03/Kayak-Logo.png" alt="Image Droite" style="width: 400px; height: auto;">
</div>

<h1 style="font-size: 18px; text-align: center; font-style: italic;">
    <span style="font-weight: bold; color: #FF6A06">
        Student : Loic VALENTINI
    </span>
</h1>

# 0. Imports

In [23]:
# Initial imports

import requests
import pandas as pd
import json
import time
import os
from dotenv import load_dotenv
import pickle


# 1. Retrieving GPS coordinates of cities via Nominatim API

In [2]:
# Creation of the city list recommended by the marketing team

list_of_cities = ["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

In [7]:
# Using the Nominatim API to retrieve GPS coordinates for each city

url = "https://nominatim.openstreetmap.org/search" # API URL

results = []  # Empty list to collect results

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36 Edge/91.0.864.59"  # User-Agent
    }

for city in list_of_cities : # Integrating each city from the list into the API

    params = {
        "city": city,   # City name
        "format": "json",  # JSON Format
    }

    time.sleep(2) # Pause in the loop to avoid blocking - 2s between each iteration

    response = requests.get(url, params=params, headers=headers) # Definition of the request response

    if response.status_code == 200:     # If No error
        data = response.json()
        if data:                        # If the data are well collected
            results.append({            # Result append
                "Ville": city,
                "Latitude": data[0]['lat'],
                "Longitude": data[0]['lon']
            })
        else:                           # In case of error
            results.append({
                "Ville": city,          # None values for the concerned city
                "Latitude": None,
                "Longitude": None
            })
    else:                               # If Error
        print(f"Erreur {response.status_code}: Impossible to get data for {city}.") # Print of a message error that incudes the concerned city
        results.append({
            "Ville": city,              # None values for this city
            "Latitude": None,
            "Longitude": None
        })

df_35_cities_GPS = pd.DataFrame(results)   # Df_35_cities_GPS creation with the retrieved data
print(df_35_cities_GPS)                    # Print of this df

if not os.path.exists('kayak_results'):    # Folder creation if missing 
    os.makedirs('kayak_results')

df.to_csv("kayak_results/35_cities_GPS.csv", index=True) # csv recording into the folder

print("CSV file recorded with index !")     # Display of a success message !

                           Ville            Latitude           Longitude
0              Mont Saint Michel          48.6359541  -1.511459954959514
1                        St Malo           48.649518          -2.0260409
2                         Bayeux          49.2764624          -0.7024738
3                       Le Havre          49.4938975           0.1079732
4                          Rouen          49.4404591           1.0939658
5                          Paris          48.8588897  2.3200410217200766
6                         Amiens          49.8941708           2.2956951
7                          Lille          50.6365654           3.0635282
8                     Strasbourg           48.584614           7.7507127
9   Chateau du Haut Koenigsbourg          48.2495226           7.3454923
10                        Colmar          48.0777517           7.3579641
11                     Eguisheim          48.0447968           7.3079618
12                      Besancon          47.238022

<h1 style="font-size: 18px;">
    This list of GPS coordinates will allow us to put them in a weather API to retrieve weather forecast. It can be found in kayak_results/35_cities_GPS.csv
</h1>

# 2. Retrieving 7-day weather forecast via the Meteomatics API

In [8]:
# Necessary libraires to import

import datetime as dt
import meteomatics.api as api

In [120]:
# List creation from the dataframe df_35_cities_GPS, coming from the csv 35_cities_GPS

import pandas as pd

# Replace 'file_path.csv' with your actual CSV file path
df_35_cities_GPS = pd.read_csv('kayak_results/35_cities_GPS.csv')

latitudes_list = df_35_cities_GPS['Latitude'].tolist()
longitudes_list = df_35_cities_GPS['Longitude'].tolist()
cities_list = df_35_cities_GPS['Ville'].tolist()

In [10]:
# Use of environment variables (username and password)

load_dotenv()

True

In [12]:
# Using the Meteomatics API to retrieve 7-days weather forecasts

username = os.environ["METEOMATICS_KEY"]
password = os.environ["METEOMATICS_SECRET_KEY"]

                                                                            # List of retried parameters (10 max. with the free API key)
parameters = ['t_2m:C', 't_max_2m_24h:C', 't_min_2m_24h:C',                 # Temperature °C (at each time point, and min/max for the previous 24 hours)
              'msl_pressure:hPa',                                           # Pressure hPa (at each time point)
              'precip_1h:mm', 'precip_24h:mm',                              # Rain mm (previous hour and previous 24hours)
              'weather_symbol_1h:idx', 'weather_symbol_24h:idx' ,           # Weather symbol index ("giving an overall impression of the weather state", previous hour and previous 24hours)
              'uv:idx',                                                     # UV index (at each time point)
              'wind_speed_10m:ms']                                          # Windforce in m/s (at each time point)


model = 'mix'                                                               # Combine of different models (intellignet blend), recommended by meteomatics

startdate = dt.datetime.utcnow().replace(minute=0, second=0, microsecond=0) # Start date = When the API is called
enddate = startdate + dt.timedelta(days=7)                                  # End date = 7 days after the Start date
interval = dt.timedelta(hours=1)                                            # 1 point per hour (expect 7*24 = 168 points per parameter and per city)


dataframes = {}                     # Creation of result dictionnary to retrieved all the collected data

if not os.path.exists('kayak_results/weather_data'):    # Folder creation if missing
    os.makedirs('kayak_results/weather_data')


for city, lat, lon in zip(cities_list, latitudes_list, longitudes_list):    # Loop for each city
    city_weather_data = api.query_time_series([(lat, lon)], startdate, enddate, interval, parameters, username, password, model)     # Launch the meteomatics API

    time.sleep(2)                   # Pause in the loop to avoid blocking - 2s between each iteration

    if city_weather_data is not None and not city_weather_data.empty: # Save the data only if the result is correct

        df_weather_city = pd.DataFrame(city_weather_data).reset_index(drop=False).reset_index(drop=True)   # Create the dataframe for each city, reset the index to columns, then add a new index starting from 0.
        dataframes[city] = df_weather_city                                                           # Place each dataframe in the dictionnary
        globals()[f"df_{city}"] = df_weather_city                                                    # Création of a dynamic variable including city name 
        df_weather_city['city'] = city                                                               # Adding of a column "city" in each df 
        df_weather_city.to_csv(os.path.join('kayak_results/weather_data', f"{city}_weather_data.csv"), index=False)   # CSV storage of each dataframe




<h1 style="font-size: 18px;">
    All the necessary data for the 35 cities has been collected. It will be cleaned and stored in usable tables to facilitate the selection of the final 5 cities. It can be found in kayak_results/weather_data/
</h1>

# 3. Cleaning and visualizing the API-collected data to determine which cities to keep

<h1 style="font-size: 18px;">

Selection Criteria for the 5 "Best-Weather" Cities:

- Limited rain in the next 7 days (if max_rain exceeds 1mm at any point during the week, the city is eliminated). Rain is measured between 7 AM and 11 PM, so nighttime rain is not considered.
- Cities are ranked by average temperature (higher is better, as it's more comfortable in winter).
- Cities are ranked by average UV index (sun exposure, higher is better). UV is measured between 8 AM and 8 PM (sunrise-sunset)
- The two rankings are multiplied, and the 5 cities with the highest scores are selected.
</h1>

In [1]:
# Recreating a weather dataframe for each city from the stored CSV files to avoid redundant API calls

import glob
import pandas as pd

filepath = 'kayak_results/weather_data/'

list_of_cities = ["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

# Loop to re-create a df per city, named df_city

for city in list_of_cities:
    
    filename = f"{city}_weather_data.csv"

    # Complete filepath
    complete_filepath = filepath + filename
    
    # Corresponding dataframe
    globals()[f"df_{city}"] = pd.read_csv(complete_filepath)

In [2]:
# Transform all validate columns into datetime format

for city in list_of_cities:
    df_city = globals().get(f"df_{city}")
    df_city["validdate"] = pd.to_datetime(df_city["validdate"])


In [3]:
# Dataframe creation about rain measurements. 3 columns : city_name, avg_rain and max_rain between 7AM/11PM


rain_df = pd.DataFrame(columns=['city_name', 'avg_rain', 'max_rain'])


for city in list_of_cities:
    df_city_loop = globals().get(f"df_{city}")  # Collect each city dataframe

    if df_city_loop is not None and not df_city_loop.empty:

        # Time filter
        df_city_loop_filtered = df_city_loop[(df_city_loop['validdate'].dt.hour >= 7) & (df_city_loop['validdate'].dt.hour < 23)]

        # Rain average
        avg_rain = df_city_loop_filtered["precip_1h:mm"].mean()

        # Rain max
        max_rain = df_city_loop_filtered["precip_1h:mm"].max()

        # Create a temporary DataFrame to append
        temporary_df = pd.DataFrame({'city_name': [city], 'avg_rain': [avg_rain], 'max_rain': [max_rain]})

        # Concatenate the temporary DataFrame with the main DataFrame
        rain_df = pd.concat([rain_df, temporary_df], ignore_index=True)


# Print the results
rain_df

C:\Users\valen\AppData\Local\Temp\ipykernel_12260\976803323.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rain_df = pd.concat([rain_df, temporary_df], ignore_index=True)


,city_name,avg_rain,max_rain
0,Mont Saint Michel,0.033186,0.49
1,St Malo,0.051770,0.61
2,Bayeux,0.015487,0.40
3,Le Havre,0.015664,0.25
4,Rouen,0.026283,0.39
5,Paris,0.043186,0.55
6,Amiens,0.019204,0.34
7,Lille,0.014602,0.20
8,Strasbourg,0.096106,0.70
9,Chateau du Haut Koenigsbourg,0.112212,0.81


In [4]:
# Dataframe creation about temperature measurements

temperature_df = pd.DataFrame(columns=['city_name', 'avg_temp', 'max_temp'])


for city in list_of_cities:
    df_city_loop = globals().get(f"df_{city}")  # Collect each city dataframe

    if df_city_loop is not None and not df_city_loop.empty:

        # temp average
        avg_temp = df_city_loop["t_2m:C"].mean()

        # temp max
        max_temp = df_city_loop["t_2m:C"].max()

        # Create a temporary DataFrame to append
        temporary_df = pd.DataFrame({'city_name': [city], 'avg_temp': [avg_temp], 'max_temp': [max_temp]})

        # Concatenate the temporary DataFrame with the main DataFrame
        temperature_df = pd.concat([temperature_df, temporary_df], ignore_index=True)


# Print the results
temperature_df


C:\Users\valen\AppData\Local\Temp\ipykernel_12260\3515248455.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  temperature_df = pd.concat([temperature_df, temporary_df], ignore_index=True)


,city_name,avg_temp,max_temp
0,Mont Saint Michel,3.820710,8.5
1,St Malo,4.568639,8.7
2,Bayeux,3.562722,9.0
3,Le Havre,3.926627,7.8
4,Rouen,3.252663,9.0
5,Paris,3.027811,7.3
6,Amiens,3.007692,8.7
7,Lille,3.079290,8.7
8,Strasbourg,1.118935,6.1
9,Chateau du Haut Koenigsbourg,-0.890533,6.5


In [5]:
# Dataframe creation about uv_index measurements

uv_index_df = pd.DataFrame(columns=['city_name', 'avg_uv_index', 'max_uv_index'])


for city in list_of_cities:
    df_city_loop = globals().get(f"df_{city}")  # Collect each city dataframe

    if df_city_loop is not None and not df_city_loop.empty:

        # Time filter
        df_city_loop_filtered = df_city_loop[(df_city_loop['validdate'].dt.hour >= 8) & (df_city_loop['validdate'].dt.hour < 20)]

        # uv_index average
        avg_uv_index = df_city_loop["uv:idx"].mean()

        # uv_index max
        max_uv_index = df_city_loop["uv:idx"].max()

        # Create a temporary DataFrame to append
        temporary_df = pd.DataFrame({'city_name': [city], 'avg_uv_index': [avg_uv_index], 'max_uv_index': [max_uv_index]})

        # Concatenate the temporary DataFrame with the main DataFrame
        uv_index_df = pd.concat([uv_index_df, temporary_df], ignore_index=True)


# Print the results
uv_index_df

C:\Users\valen\AppData\Local\Temp\ipykernel_12260\3385378747.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  uv_index_df = pd.concat([uv_index_df, temporary_df], ignore_index=True)


,city_name,avg_uv_index,max_uv_index
0,Mont Saint Michel,0.213018,3.0
1,St Malo,0.213018,3.0
2,Bayeux,0.213018,3.0
3,Le Havre,0.242604,3.0
4,Rouen,0.218935,3.0
5,Paris,0.224852,3.0
6,Amiens,0.224852,3.0
7,Lille,0.213018,3.0
8,Strasbourg,0.242604,3.0
9,Chateau du Haut Koenigsbourg,0.254438,3.0


In [6]:
# collect of the GPS coordinates into  datframe from the stored CSV

df_gps = pd.read_csv('kayak_results/35_cities_GPS.csv')     # Collect the dataframe
df_gps = df_gps[['Ville','Latitude','Longitude']]           # Keep the useful columns
df_gps = df_gps.rename(columns={'Ville': 'city_name'})      # Rename the key

df_gps

,city_name,Latitude,Longitude
0,Mont Saint Michel,48.635954,-1.511460
1,St Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966
5,Paris,48.853495,2.348391
6,Amiens,49.894171,2.295695
7,Lille,50.636565,3.063528
8,Strasbourg,48.584614,7.750713
9,Chateau du Haut Koenigsbourg,48.249523,7.345492


In [7]:
# Merge of the 3 weather Dataframes + GPS

import os

merged1_df = pd.merge(rain_df, temperature_df, on='city_name', how='inner')  # Merge rain_df and temperature_df
merged2_df = pd.merge(merged1_df, uv_index_df, on='city_name', how='inner')  # Merge with uv_index_df
weather_decision_merged_df = pd.merge(merged2_df, df_gps, on='city_name', how='inner')  # Merge with df_gps

weather_decision_merged_df.to_csv(os.path.join('kayak_results', f"weather_decision.csv"), index=False)   # CSV storage of this dataframe

print("CSV file recorded !")     # Display of a success message !


CSV file recorded !


In [8]:
# Barchart of max_rain per city

import plotly.express as px

# Add a column to define the color (Red if > 1mm, Green otherwise)
weather_decision_merged_df['color'] = weather_decision_merged_df['max_rain'].apply(lambda x: 'red (>1mm)' if x > 1 else 'green (<1mm)')

# Sort the data by max_rain (from highest to lowest)
weather_decision_merged_df = weather_decision_merged_df.sort_values(by='max_rain', ascending=False)

# Create the chart with Plotly Express
fig = px.bar(weather_decision_merged_df, x='city_name', y='max_rain', 
             title="Max Rainfall per City",
             labels={'max_rain': 'Max Rain (mm)', 'city_name': 'City'},
             text='max_rain',  
             color='color',  # Color gradient based on rainfall
             color_discrete_map={'red (>1mm)': 'red', 'green (<1mm)': 'green'}  # Define the colors
            )

# Customize the title and layout
fig.update_layout(
    title={
        'text': "Max Rainfall per City",
        'x': 0.5,  # Center the title
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(family="Arial", size=20, color="black")  # Increase size + black color
    }, # Bold and styled title
    
    xaxis_title=None  # Remove the title of the X-axis
)

# Show the figure
fig.show()



In [14]:
# Same vizualisation on a France map

# Activate Mapbox (requires a Mapbox API key, but works with the public version)
px.set_mapbox_access_token("pk.eyJ1IjoibWFwYm94IiwiYSI6ImNqZ2czZzYzaTAwN3kyeHFwbXExdGVvbjYifQ.CqHfA9sT2IRKIIeOkeV6GA")

# Add a column to define the color (Red if > 1mm, Green otherwise)
weather_decision_merged_df['color'] = weather_decision_merged_df['max_rain'].apply(lambda x: 'red (>1mm)' if x > 1 else 'green (<1mm)')

# Create the map with Plotly Express
fig = px.scatter_mapbox(
    weather_decision_merged_df,
    lat='Latitude',
    lon='Longitude',
    color='color',  # Custom colors
    size='max_rain',  # Size of the points proportional to max_rain
    size_max=20,  # Max size of the points
    hover_name='city_name',
    hover_data={'Latitude': False, 'Longitude': False, 'max_rain': True},  # Hover info
    color_discrete_map={'red (>1mm)': 'red', 'green (<1mm)': 'green'},  # Custom legend
    title="Rainfall Map of France"
)

# Customize the display
fig.update_layout(
    height=800,
    width = 1000,
    mapbox_zoom=5,
    mapbox_style="carto-positron",  # Map style
    mapbox_center={"lat": 46.6031, "lon": 1.8883},  # Center on France
    title={
        'text': "Rainfall Map of France",
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=20, color="black")
    }
)

# Display the map
fig.show()



In [22]:
# Store this map in html

import plotly.express as px
import plotly.io as pio

# Save this map in HTML format instead of JPEG
fig.write_html("kayak_results/rainfall_cities_elimination.html")
print("HTML export completed successfully!")



HTML export completed successfully!


In [80]:
# Removal of the 10 cities that did not meet the rain criteria.

weather_decision_filtered_df = weather_decision_merged_df[weather_decision_merged_df['color'] == 'green (<1mm)']
list_of_cities = weather_decision_filtered_df['city_name'].tolist()

len(list_of_cities) # Check that the list is limited to 25 cities

25

In [85]:
# Obtaining the ranking based on the temperature and UV index criteria

# Make a copy of the sliced DataFrame to avoid the warning
weather_decision_filtered_df = weather_decision_filtered_df.copy()

# Create a new column 'rank_temp' based on the 'avg_temp' column in descending order
weather_decision_filtered_df['rank_temp'] = weather_decision_filtered_df['avg_temp'].rank(ascending=False, method='min').astype(int)

# Create a new column 'rank_uv_index' based on the 'avg_uv_index' column in descending order
weather_decision_filtered_df['rank_uv_index'] = weather_decision_filtered_df['avg_uv_index'].rank(ascending=False, method='min').astype(int)

# Sum of these two columns
weather_decision_filtered_df['overall_rating'] = weather_decision_filtered_df['rank_temp'] + weather_decision_filtered_df['rank_uv_index'].astype(int)

# Final ranking accorind to this overall_rating, in ascending order
weather_decision_filtered_df['final_ranking'] = weather_decision_filtered_df['overall_rating'].rank(ascending=True, method='min').astype(int)



In [ ]:
# Sort the dataframe et see the Top5 cities

weather_decision_filtered_df = weather_decision_filtered_df.sort_values(by='final_ranking', ascending=True)
weather_decision_filtered_df[['city_name','final_ranking']]


,city_name,final_ranking
30,Toulouse,1
21,Aix en Provence,2
31,Montauban,2
18,Bormes les Mimosas,2
20,Marseille,5
28,Carcassonne,6
19,Cassis,6
17,Gorges du Verdon,8
15,Grenoble,9
34,La Rochelle,10


In [189]:
# Same vizualisation on a France map


# Add a column to define the color (Red if > 1mm, Green otherwise)
weather_decision_filtered_df['color'] = weather_decision_filtered_df['final_ranking'].apply(lambda x: 'red (NOT TOP5)' if x > 5 else 'green (TOP5)')

# Create the map with Plotly Express
fig = px.scatter_mapbox(
    weather_decision_filtered_df,
    lat='Latitude',
    lon='Longitude',
    color='color',  # Custom colors
    size=[10] * len(weather_decision_filtered_df),
    hover_name='city_name',
    hover_data={'Latitude': False, 'Longitude': False, 'final_ranking': True},  # Hover info
    color_discrete_map={'red (NOT TOP5)': 'red', 'green (TOP5)': 'green'},  # Custom legend
    title="TOP5 Cities of France",
)

# Customize the display
fig.update_layout(
    height=800,
    width = 1000,
    mapbox_zoom=5,
    mapbox_style="carto-positron",  # Map style
    mapbox_center={"lat": 46.6031, "lon": 1.8883},  # Center on France
    title={
        'text': "TOP5 Cities of France",
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=20, color="black")
    }
)

# Display the map
fig.show()

# Store this map in jpeg

fig.write_image("kayak_results/TOP5_cities.jpg", format="jpeg")

print("JPEG storage Done !")

JPEG storage Done !


In [97]:
# Final list of cities
 
TOP5_list_of_cities = weather_decision_filtered_df[weather_decision_filtered_df['final_ranking'] < 6]['city_name'].tolist()

TOP5_list_of_cities # To see the list


['Toulouse', 'Aix en Provence', 'Montauban', 'Bormes les Mimosas', 'Marseille']

<h1 style="font-size: 18px;">
    The TOP5 cities have been well defined, and now we can proceed with scraping data from Booking.com!
</h1>


# 4. Scraping a list of 25 hotel URLs per city

<h1 style="font-size: 18px;">

The goal of this section is to scrape the URLs of various hotel per city, so that we can later scrape all the necessary information from this list of URLs 
</h1>

In [24]:
# Recreate the list of cities to avoid rerunning all previous cells and continue progressing."

import pickle

# Load the variable from the saved file
with open('TOP5_list_of_cities.pkl', 'rb') as f:
    TOP5_list_of_cities = pickle.load(f)

# After a quick check on Booking.com, spaces in the city names are replaced with '-' in the URLs.

TOP5_list_of_cities = [city.replace(" ", "-") for city in TOP5_list_of_cities]
TOP5_list_of_cities



['Toulouse', 'Aix-en-Provence', 'Montauban', 'Bormes-les-Mimosas', 'Marseille']

In [25]:
# Make this variable available for the python spider

import pickle

# Save the variable to a file
with open('TOP5_list_of_cities.pkl', 'wb') as f:
    pickle.dump(TOP5_list_of_cities, f)


In [103]:
# Launch the Spider1_KAYAK to scrap the data

!python Spider1_KAYAK.py

2025-02-08 22:20:32 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: scrapybot)
2025-02-08 22:20:32 [scrapy.utils.log] INFO: Versions: lxml 4.9.3.0, libxml2 2.10.4, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 23.10.0, Python 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)], pyOpenSSL 24.0.0 (OpenSSL 3.0.15 3 Sep 2024), cryptography 42.0.2, Platform Windows-10-10.0.19045-SP0
2025-02-08 22:20:32 [scrapy.addons] INFO: Enabled addons:
[]
2025-02-08 22:20:32 [py.warnings] WARNING: c:\Users\valen\anaconda3\Lib\site-packages\scrapy\utils\request.py:254: ScrapyDeprecationWarning: '2.6' is a deprecated value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting.

It is also the default value. In other words, it is normal to get this warning if you have not defined a value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting. This is so for backward compatibility reasons, but it will change in a future version of Scrapy.

See the docu

In [1]:
# Transformation of the JSON into a Dataframe

import json
import pandas as pd


filepath = 'kayak_results/List_of_URLs_per_TOP5_cities.json'

# Reading of the JSON
with open(filepath, 'r', encoding='utf-8') as file:
    hotels_url = json.load(file)

# Into a dataframe for a better visualization
df_hotels_url = pd.DataFrame(hotels_url)

df_hotels_url

,city,hotel_names,hotel_urls
0,Montauban,"[Brit Hotel Confort Montauban, Appartement Cos...",[https://www.booking.com/hotel/fr/deltour-mont...
1,Marseille,"[The Babel Community Hôtel - Vieux Port, Toyok...",[https://www.booking.com/hotel/fr/grand-t1-vie...
2,Aix-en-Provence,"[thecamp Hôtel & Lodges - Aix en Provence, Séj...",[https://www.booking.com/hotel/fr/thecamp-amp-...
3,Bormes-les-Mimosas,[Cosy 36m avec jolie VUE MER centre-village !...,[https://www.booking.com/hotel/fr/cosy-36m-wit...
4,Toulouse,"[The Social Hub Toulouse, Odalys City Toulouse...",[https://www.booking.com/hotel/fr/the-social-h...


In [8]:
# Cleaning to get a readable list of hotel_urls

df_hotels_url_exploded = df_hotels_url[['city','hotel_names','hotel_urls']].explode(['hotel_names','hotel_urls'], ignore_index=True)
list_of_urls = df_hotels_url_exploded.iloc[:, 1].tolist() 


In [26]:
# Make this variable available for the python spider

# Save the variable to a file
with open('list_of_urls.pkl', 'wb') as f:
    pickle.dump(list_of_urls, f)

NameError: name 'list_of_urls' is not defined

<h1 style="font-size: 18px;">
    Now that we have gathered a list of 25 hotel URLs per city, we will be able to scrape various details from these hotels! It can be found in kayak_result/list_of_urls.pkl
</h1>

# 5. Scraping all the hotel details

In [ ]:
# Recreate the list of cities to avoid rerunning all previous cells and continue progressing."

import pickle

# Load the variable from the saved file
with open('list_of_urls.pkl', 'rb') as f:
    list_of_urls = pickle.load(f)


FileNotFoundError: [Errno 2] No such file or directory: 'kayak_results/list_of_urls.pkl'

In [112]:
# Launch the Spider2_KAYAK to scrap the data from the list_of_urls

!python Spider2_KAYAK.py

2025-02-08 22:39:17 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: scrapybot)
2025-02-08 22:39:17 [scrapy.utils.log] INFO: Versions: lxml 4.9.3.0, libxml2 2.10.4, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 23.10.0, Python 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)], pyOpenSSL 24.0.0 (OpenSSL 3.0.15 3 Sep 2024), cryptography 42.0.2, Platform Windows-10-10.0.19045-SP0
2025-02-08 22:39:17 [scrapy.addons] INFO: Enabled addons:
[]
2025-02-08 22:39:17 [py.warnings] WARNING: c:\Users\valen\anaconda3\Lib\site-packages\scrapy\utils\request.py:254: ScrapyDeprecationWarning: '2.6' is a deprecated value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting.

It is also the default value. In other words, it is normal to get this warning if you have not defined a value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting. This is so for backward compatibility reasons, but it will change in a future version of Scrapy.

See the docu

In [4]:
# Transformation of the JSON into a Dataframe

import json
import pandas as pd


filepath = 'kayak_results/hotels_details.json'

# Reading of the JSON
with open(filepath, 'r', encoding='utf-8') as file:
    hotels_details = json.load(file)

# Into a dataframe for a better visualization
df_hotel_details = pd.DataFrame(hotels_details)

df_hotel_details

,hotel_url,hotel_gps,hotel_desc,hotel_mark
0,https://www.booking.com/hotel/fr/le-point-virg...,"44.0222532,1.3591095","[Situé à Montauban, l’hébergement Le Point Vir...",None
1,https://www.booking.com/hotel/fr/le-beau-malco...,"44.0173306,1.3534844",[Le beau Malcousinat proche place Nationale es...,"9,1"
2,https://www.booking.com/hotel/fr/appartement-c...,"44.0191236,1.3564167",[L'Appartement Cosy en Centre ville est situé ...,"8,2"
3,https://www.booking.com/hotel/fr/dali-montauba...,"44.020995559797,1.349240014017","[Situé à Montauban, le Dali Hôtel Montauban pr...","9,0"
4,https://www.booking.com/hotel/fr/appartement-d...,"44.01682794568838,1.3424564997115507","[Situé à Montauban, l’hébergement L’appartemen...","9,1"
...,...,...,...,...
120,https://www.booking.com/hotel/fr/campanilepurp...,"43.606810036550876,1.3937294483184814",[Cet hôtel Campanile est situé à proximité de ...,"8,5"
121,https://www.booking.com/hotel/fr/residhotel-to...,"43.59539161058483,1.4412049948077765",[L’établissement Residhotel Toulouse Centre se...,"7,9"
122,https://www.booking.com/hotel/fr/cousture.fr.h...,"43.604250797317704,1.4504990726709366",[L’Hôtel Le Cousture vous accueille au cœur de...,"7,6"
123,https://www.booking.com/hotel/fr/le-clos-des-s...,"43.594413616900646,1.44607275724411",[Situé au cœur du centre-ville historique de T...,"9,6"


In [9]:
# Merge of this dataframe with the previous one to get the city_name and data transformation

df_kayak_final = pd.merge(df_hotels_url_exploded, df_hotel_details, left_on='hotel_urls', right_on='hotel_url', how='inner')

# Separation of longitude / latitude
df_kayak_final[['hotel_latitude', 'hotel_longitude']] = df_kayak_final['hotel_gps'].str.split(',', expand=True)

# Keeping of the interesting columns
df_kayak_final = df_kayak_final[['city', 'hotel_names', 'hotel_longitude', 'hotel_latitude', 'hotel_url', 'hotel_desc', 'hotel_mark']]

# Type transformation as float
df_kayak_final['hotel_latitude'] = pd.to_numeric(df_kayak_final['hotel_latitude'], errors='coerce')
df_kayak_final['hotel_longitude'] = pd.to_numeric(df_kayak_final['hotel_longitude'], errors='coerce')

df_kayak_final['hotel_mark'] = df_kayak_final['hotel_mark'].str.replace(',', '.')  # Replace commas
df_kayak_final['hotel_mark'] = pd.to_numeric(df_kayak_final['hotel_mark'], errors='coerce')



df_kayak_final

,city,hotel_names,hotel_longitude,hotel_latitude,hotel_url,hotel_desc,hotel_mark
0,Montauban,Brit Hotel Confort Montauban,1.330394,43.980811,https://www.booking.com/hotel/fr/deltour-monta...,[L’établissement Brit Hotel Confort Montauban ...,8.2
1,Montauban,Appartement Cosy en Centre ville,1.356417,44.019124,https://www.booking.com/hotel/fr/appartement-c...,[L'Appartement Cosy en Centre ville est situé ...,8.2
2,Montauban,Sure Hotel by Best Western Les Portes de Monta...,1.322098,43.936287,https://www.booking.com/hotel/fr/sure-by-best-...,"[Situé à Montauban, à 42 km de l'amphithéâtre ...",8.8
3,Montauban,Hôtel du Commerce,1.354073,44.015426,https://www.booking.com/hotel/fr/hotel-du-comm...,"[Situé dans le centre de Montauban, l'Hôtel du...",8.4
4,Montauban,Dali Hôtel Montauban,1.349240,44.020996,https://www.booking.com/hotel/fr/dali-montauba...,"[Situé à Montauban, le Dali Hôtel Montauban pr...",9.0
...,...,...,...,...,...,...,...
120,Toulouse,Campanile Toulouse Purpan,1.393729,43.606810,https://www.booking.com/hotel/fr/campanilepurp...,[Cet hôtel Campanile est situé à proximité de ...,8.5
121,Toulouse,Pénitents by Yumē,1.450010,43.601988,https://www.booking.com/hotel/fr/penitents-by-...,"[Situé dans le centre de Toulouse, à 3,8 km du...",8.5
122,Toulouse,Residhotel Toulouse Centre,1.441205,43.595392,https://www.booking.com/hotel/fr/residhotel-to...,[L’établissement Residhotel Toulouse Centre se...,7.9
123,Toulouse,Hôtel Le Cousture,1.450499,43.604251,https://www.booking.com/hotel/fr/cousture.fr.h...,[L’Hôtel Le Cousture vous accueille au cœur de...,7.6


In [12]:
# Storage of this important table in a CSV

df_kayak_final.to_csv(os.path.join('kayak_results', f"kayak_final_details.csv"), index=False)

<h1 style="font-size: 18px;">
    The dataframe df_kayak_final contains all the expected information and has been stored in a CSV file named kayak_results/kayak_final_details.
</h1>

<h1 style="font-size: 18px;">
    Let's assume we want to view the top 20 highest-ranked hotels from the entire scraped list
</h1>

In [13]:
# Filtering the non-marked hotel (must not be considered in our case)

df_kayak_final = df_kayak_final[df_kayak_final['hotel_mark'].notnull()]

In [14]:
# Filtering the non-marked hotel (must not be considered in our case)

df_kayak_TOP20_hotels = df_kayak_final.sort_values(by='hotel_mark', ascending=False).head(20)
df_kayak_TOP20_hotels

,city,hotel_names,hotel_longitude,hotel_latitude,hotel_url,hotel_desc,hotel_mark
77,Bormes-les-Mimosas,SELECT'soHOME - Superbe appartement pour 4 per...,6.359008,43.124624,https://www.booking.com/hotel/fr/select-sohome...,[Situé à 700 mètres de la plage de la Pointe d...,10.0
13,Montauban,Villa de cammas,1.317364,44.013202,https://www.booking.com/hotel/fr/villa-de-camm...,"[Située à Montauban, en Midi-Pyrénées, la Vill...",9.8
14,Montauban,L'authentique,1.355709,44.017520,https://www.booking.com/hotel/fr/authentique-m...,[L'authentique est situé à Montauban. Une conn...,9.7
41,Marseille,Résidence Kley Marseille République,5.369920,43.301093,https://www.booking.com/hotel/fr/residence-kle...,[La Résidence Kley Marseille République est si...,9.7
124,Toulouse,Le Clos des Salins,1.446073,43.594414,https://www.booking.com/hotel/fr/le-clos-des-s...,[Situé au cœur du centre-ville historique de T...,9.6
88,Bormes-les-Mimosas,The Little House,6.342202,43.150361,https://www.booking.com/hotel/fr/the-little-ho...,"[Situé à Bormes-les-Mimosas, à 29 km du châtea...",9.6
81,Bormes-les-Mimosas,Cosy studio - Place du Figuier,6.342968,43.151455,https://www.booking.com/hotel/fr/studio-figuie...,"[Situé à Bormes-les-Mimosas, à 29 km du châtea...",9.6
11,Montauban,La Halte Occitane - Classé 3 étoiles - Pour 6 ...,1.315446,44.015068,https://www.booking.com/hotel/fr/beau-t4-tout-...,"[Situé à Montauban, l’hébergement La Halte Occ...",9.6
42,Marseille,Élégant appartement - centre Prado Vélodrome,5.387800,43.269802,https://www.booking.com/hotel/fr/elegant-appar...,"[Situé à Marseille, à 1,7 km de la plage de l'...",9.6
90,Bormes-les-Mimosas,SELECT' - T2 avec vue - Piscine et parking pri...,6.352916,43.142718,https://www.booking.com/hotel/fr/select-t2-ave...,[L’hébergement SELECT' - T2 avec vue - Piscine...,9.5


In [17]:
# Visualize the TOP20Hotels in a French map

# Adjust the size of the markers based on 'hotel_mark'
size_max = 30 
min_size = 5 
df_kayak_TOP20_hotels['normalized_size'] = (df_kayak_TOP20_hotels['hotel_mark'] - df_kayak_TOP20_hotels['hotel_mark'].min()) / (df_kayak_TOP20_hotels['hotel_mark'].max() - df_kayak_TOP20_hotels['hotel_mark'].min()) * (size_max - min_size) + min_size

# Create the map with Plotly Express
fig = px.scatter_mapbox(
    df_kayak_TOP20_hotels,
    lat='hotel_latitude',
    lon='hotel_longitude',
    size='normalized_size',  # Size based on the normalized hotel_mark
    hover_name='city',
    hover_data={'hotel_latitude': False, 'hotel_longitude': False, 'hotel_mark': True},  # Hover info
    title="TOP20 Hotels of TOP5 cities",
)

# Customize the display
fig.update_layout(
    height=800,
    width=1000,
    mapbox_zoom=6.5,
    mapbox_style="carto-positron",  # Map style
    mapbox_center={"lat": 43.62, "lon": 3.86},  # relevant place defined by testing
    title={
        'text': "TOP20 Hotels of TOP5 cities",
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=20, color="black")
    }
)

# Display the map
fig.show()

# Store this map in jpeg

fig.write_image("kayak_results/TOP20_Hotels_of_TOP5_cities.jpg", format="jpeg")

print("JPEG storage Done !")


JPEG storage Done !


In [188]:
# As the map is wide, let's zoom on one city to ensure this is correct

# Adjust the size of the markers based on 'hotel_mark'
size_max = 30 
min_size = 5 
df_kayak_TOP20_hotels['normalized_size'] = (df_kayak_TOP20_hotels['hotel_mark'] - df_kayak_TOP20_hotels['hotel_mark'].min()) / (df_kayak_TOP20_hotels['hotel_mark'].max() - df_kayak_TOP20_hotels['hotel_mark'].min()) * (size_max - min_size) + min_size

# Create the map with Plotly Express
fig = px.scatter_mapbox(
    df_kayak_TOP20_hotels,
    lat='hotel_latitude',
    lon='hotel_longitude',
    size='normalized_size',  # Size based on the normalized hotel_mark
    hover_name='city',
    hover_data={'hotel_latitude': False, 'hotel_longitude': False, 'hotel_mark': True},  # Hover info
    title="TOP20 Hotels of Marseille",
)

# Customize the display
fig.update_layout(
    height=800,
    width=1000,
    mapbox_zoom=12,
    mapbox_style="carto-positron",  # Map style
    mapbox_center={"lat": 43.30, "lon": 5.4},  # relevant place defined by testing
    title={
        'text': "TOP20 Hotels of Marseille",
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=20, color="black")
    }
)

# Display the map
fig.show()

# Store this map in jpeg

fig.write_image("kayak_results/Final_example_on_Marseille.jpg", format="jpeg")

print("JPEG storage Done !")

JPEG storage Done !


<h1 style="font-size: 18px;">
    We have successfully generated the map displaying the 20 selected hotels, with a zoom-in on the city of Marseille to ensure optimal visualization
</h1>

# 6. Placing the dataset "kayak_final_details.csv" into AWS S3

<h1 style="font-size: 18px;">
    A specific S3 bucket called 'kayak-project-loicvalentini' has been created.
</h1>

<a href="https://ibb.co/nNrbKggB"><img src="https://i.ibb.co/PvxNVmmM/S3-Bucket.jpg" alt="S3-Bucket" border="0" /></a>

<h1 style="font-size: 18px;">
    The dataset 'kayak_final_details.csv' has been uploaded to this new bucket via drag and drop.
</h1>

<a href="https://ibb.co/ch21Gv28"><img src="https://i.ibb.co/23cyD6cM/S3-File.jpg" alt="S3-File" border="0" /></a>

<h1 style="font-size: 18px;">
    But also with python code.
</h1>

In [1]:
# Necessary libraries to import

import boto3
from dotenv import load_dotenv
import os
import pandas as pd
load_dotenv()

True

In [2]:
# AWS Session to open

session = boto3.Session(aws_access_key_id=os.getenv("AWS_KEY"), 
                        aws_secret_access_key=os.getenv("AWS_SECRET_KEY"))

In [8]:
# Creation of the necessary S3 variables

s3 = session.resource('s3')
bucket_name = "kayak-project-loicvalentini"
bucket = s3.Bucket(bucket_name)

In [ ]:
# Reimport of the kayak_final_results data and generation of a CSV variable

df_kayak_final_results = pd.read_csv('kayak_results/kayak_final_details.csv')
df_kayak_final_results.csv = df_kayak_final_results.to_csv()

In [ ]:
# CSV file to put into the S3 bucket

put_object = bucket.put_object(Key="df_kayak_final_results.csv", Body=df_kayak_final_results.csv)

<a href="https://ibb.co/DH3mNv00"><img src="https://i.ibb.co/4nBCcXhh/S3-Bucket-code-version.jpg" alt="S3-Bucket-code-version" border="0" /></a>

<h1 style="font-size: 18px;">
    Next, let's make it available in an SQL database.
</h1>

# 7.  Availability of the dataset on AWS RDS

<h1 style="font-size: 18px;">
    A specific RDS database bucket called 'database-kayak-details-loicvalentini' has been created.
</h1>

<a href="https://ibb.co/qLYvG54L"><img src="https://i.ibb.co/p6jSDys6/RDS-Database.jpg" alt="RDS-Database" border="0" /></a>

<h1 style="font-size: 18px;">
    Now, let's set up a functional database.
</h1>

In [1]:
# Necessary imports

from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import boto3
import os 
import pandas as pd
from io import StringIO

load_dotenv()



True

In [ ]:
# Retrieving the csv file from S3

# Creation of s3 client

s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv("AWS_KEY"), 
    aws_secret_access_key=os.getenv("AWS_SECRET_KEY")
)

# Uploading the file from s3

bucket_name = 'kayak-project-loicvalentini'
file_key = 'kayak_final_details.csv'

response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
file_content = response['Body'].read().decode('utf-8')


In [ ]:
# Engine creation

engine = create_engine(f"postgresql+psycopg2://{os.getenv('POSTGRE_USERNAME')}:{os.getenv('POSTGRE_PASSWORD')}@{os.getenv('POSTGRE_HOSTNAME')}", echo=True)

In [ ]:
# Replacing this .csv by a pandas Dataframe

df_kayak = pd.read_csv(StringIO(file_content))


In [ ]:
# Insertion of this data into RDS

df_kayak.to_sql('database_kayak_details', con=engine, if_exists='replace', index=False)

2025-02-10 20:31:53,713 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-02-10 20:31:53,714 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-02-10 20:31:53,731 INFO sqlalchemy.engine.Engine select current_schema()
2025-02-10 20:31:53,733 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-02-10 20:31:53,750 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-02-10 20:31:53,751 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-02-10 20:31:53,778 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-02-10 20:31:53,789 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname

125

<h1 style="font-size: 18px;">
    This table can also be viewed in PgAdmin, which is connected to RDS.
</h1>

<a href="https://ibb.co/zVPfXwsk"><img src="https://i.ibb.co/3mShpH1J/Pg-Admin-Table.jpg" alt="Pg-Admin-Table" border="0" /></a>

<h1 style="font-size: 18px;">
    It is now fully ready for use to perform further analysis !
</h1>